## NASA Turbofan Engine Degradation Simulation-2

### Introducción:

En el año 2008, el dataset "Turbofan Engine Degradation Simulation" se lanza como concurso durante la Primera "International Conference on Prognostics and Health Management (PHM08)", y cuya data fue generada mediante C-MAPSS, una herramienta de MATLAB/Simulink desarrollada por la NASA para simular motores turbofan comerciales. Este dataset rápidamente se convirtió en el "hello world" para probar algoritmos de mantenimiento predictivo, al ser citado en más de 1000 investigaciones.

Consecuencia de este éxito, en el 2021 se publica el dataset N-CMAPSS (New Commercial Modular Aero-Propulsion System Simulation), que es la evolución directa del C-MAPSS del 2008. 


El N-CMAPSS extiende la data original al incorporar condiciones de vuelo reales derivadas de datos operacionales de aviones comerciales, aumentando sustancialmente la escala del dataset a millones de muestras, y por lo tanto ya no se trata de simulación abstracta: cada vuelo contiene grabaciones que cubren condiciones de ascenso, vuelo de crucero y descenso, correspondientes a diferentes rutas operadas por aviones.

El carácter realista del presente dataset y su orientación al campo de la prognosis o mantenimiento predictivo lo hace de gran interés para los presentes autores, que cuentan con la especialidad de ingenieros técnicos industriales.



### Definición del problema:

El sistema bajo análisis corresponde a un motor turbofán comercial de doble eje y alto bypass. El motor consta de seis componentes principales: ventilador (fan), compresor de baja presión (LPC), compresor de alta presión (HPC), cámara de combustión, turbina de alta presión (HPT) y turbina de baja presión (LPT). A lo largo de su vida operativa, estos componentes experimentan un proceso de degradación progresiva e irreversible. Uno de los problemas principales es la degradación de la eficiencia de la turbina de alta presión (HPT-E), cuyo deterioro se manifiesta de forma inicialmente imperceptible en las señales de los sensores y se acelera gradualmente conforme el motor se aproxima al fin de su vida útil.

El problema que se plantea resolver es estimar la Vida Útil Remanente (Remaining Useful Life, RUL) del motor, siendo el objetivo final desarrollar un modelo de regresión capaz de predecir con precisión el RUL a partir del historial de señales de sensores, contribuyendo así a una estrategia de mantenimiento predictivo que permita programar intervenciones antes del fallo, reducir el tiempo de inactividad no planificado y optimizar los costes operativos de la flota.

### Marco teórico:

El mantenimiento predictivo constituye una estrategia de gestión de activos que busca anticipar el fallo de un componente o sistema antes de que ocurra, optimizando así los intervalos de intervención y reduciendo tanto el tiempo de inactividad no planificado como los costes operativos.

En este contexto surge el campo de la Gestión del Pronóstico y la Salud (Prognostics and Health Management, PHM), cuyo objetivo es integrar la adquisición de datos de sensores, el procesamiento de señales, la detección de anomalías y la estimación de la vida útil remanente en un flujo de trabajo unificado orientado a la toma de decisiones de mantenimiento.

Vida Útil Remanente (RUL)

La Vida Útil Remanente (Remaining Useful Life, RUL) es la variable objetivo central en los problemas de pronóstico de degradación. Se define formalmente como el número de ciclos operativos —o unidades de tiempo— que le restan a un componente antes de alcanzar un umbral de fallo predefinido, dado su estado de degradación actual.

Modelos de Datos para el Pronóstico de Degradación:

Los enfoques de modelado en PHM se clasifican en tres grandes categorías:

- Modelos basados en física
- Modelos basados en datos
- Modelos híbridos

El presente análisis se enmarca en el enfoque basado en datos, cuya cadena de procesamiento típica comprende: adquisición de señales de sensores → preprocesamiento y selección de características → entrenamiento del modelo de regresión → evaluación mediante métricas de error.

### Metodología usada:

El proyecto se compone de **cinco fases**, desde la carga del archivo HDF5 hasta la evaluación del modelo entrenado:

Carga de datos → Análisis exploratorio → Feature Engineering → Entrenamiento → Evaluación

El análisis exploratorio (EDA) comprende lo siguiente:

- Distribución del RUL y trayectorias de degradación por unidad
- Evolución temporal de los sensores a lo largo de los ciclos de vuelo
- Efecto de la clase de vuelo (Fc) sobre los niveles absolutos de los sensores
- Correlación de Pearson de cada variable con el RUL, para identificar señales informativas
- Matriz de correlación inter-sensores, para detectar redundancia entre variables

El feature engineering comprende tres grupos de variables derivadas:

- **Posición intra-ciclo** (`pos_relativa`): posición normalizada [0, 1] dentro del ciclo de vuelo, que permite al modelo distinguir entre fases de despegue, crucero y descenso cuando los valores absolutos de los sensores son similares.
- **Estadísticos rolling inter-ciclo**: media y desviación típica de cada sensor en ventanas de 5 y 10 ciclos, más slope lineal en ventana de 10 ciclos. Capturan la tendencia acumulada de degradación entre vuelos.
- **Estadísticos intra-ciclo**: máximo y desviación típica del ciclo actual, que reflejan el comportamiento del sensor bajo máxima demanda y su estabilidad dentro del vuelo.

El modelo sigue una **arquitectura predecir-luego-agregar**: se genera una predicción de RUL por cada observación individual; la estimación definitiva por ciclo se obtiene como la **mediana** de dichas predicciones, lo que aporta robustez frente a los picos de señal en despegue y aterrizaje.

El entrenamiento del modelo se compone de 2 fases:

Fase 1: Optimización de hiperparámetros. Se realiza una búsqueda de hiperparámetros mediante **Optuna** (50 trials, muestreador TPE) sobre el conjunto de entrenamiento. Dado el reducido número de unidades disponibles en Fc=3, se aplica un esquema **leave-one-unit-out**: la última unidad se reserva como conjunto de validación, garantizando que ningún ciclo de la misma unidad aparezca en entrenamiento. La métrica de validación es el RMSE calculado sobre las predicciones **agregadas por ciclo (mediana)**, alineando así la optimización con el objetivo real del modelo. Los hiperparámetros sometidos a búsqueda son: `n_estimators`, `learning_rate`, `max_depth`, `subsample`, `colsample_bytree`, `min_child_weight`, `gamma`, `reg_lambda` y `reg_alpha`.

Fase 2: Entrenamiento final. Con los hiperparámetros óptimos identificados, se entrena el modelo definitivo sobre la totalidad del conjunto de desarrollo, utilizando el conjunto de validación como `eval_set` para aplicar *early stopping* y determinar el número óptimo de árboles.

Finalmente, el modelo se evalúa sobre el conjunto de prueba mediante las siguientes métricas:

- **RMSE** como métrica principal
- **MAE** para caracterizar el error típico sin amplificación de outliers
- **R²** para cuantificar la proporción de varianza del RUL explicada por el modelo
- **Score NASA (S)**: métrica asimétrica que penaliza más las predicciones tardías (RUL estimado > real) que las adelantadas, ya que predecir vida remanente cuando el motor está próximo al fallo es más peligroso que anticiparse. Se calcula como la suma de términos exponenciales con distinta base según el signo del error.
- **Análisis de incertidumbre**: basado en la dispersión intra-ciclo de las predicciones individuales (`RUL_std`), que permite identificar zonas de degradación inestable y construir un mapa de alarma operativo.

### Modelos / algoritmos utilizados:

Se utilizará XGboost como modelo de regresión para la estimación del RUL en nuestro dataset, estando justificado por las siguientes propiedades:

- Manejo de no linealidades. La degradación del motor turbofán es un proceso inherentemente no lineal. XGBoost, al basarse en árboles de decisión, captura automáticamente relaciones no lineales y discontinuidades en el espacio de características sin requerir transformaciones explícitas de las variables.

- Interacciones entre variables. Las señales de temperatura, presión y velocidad del motor interactúan de forma compleja entre sí y con las condiciones de vuelo.

- Robustez frente a características no normalizadas. A diferencia de los modelos lineales o las redes neuronales, XGBoost tiene reducida sensibilidad a la escala absoluta de los sensores.

- Importancia de características. XGBoost proporciona métricas de importancia de variables (gain, cover, frequency) que permiten identificar qué sensores contribuyen más a la predicción del RUL, facilitando la interpretabilidad del modelo y la validación del conocimiento de dominio.

- Eficiencia computacional. El algoritmo implementa construcción paralela de árboles a nivel de nodo, cómputo aproximado de divisiones mediante histogramas, y procesamiento eficiente de datos dispersos, lo que lo hace viable para el volumen de datos del dataset.

### Datos empleados:

El dataset utilizado en este proyecto es el N-CMAPSS DS01, correspondiente al archivo N-CMAPSS_DS01-005.h5, distribuido por NASA en formato HDF5 (Hierarchical Data Format v5). Según la tabla de datasets del paper oficial, DS01 contiene:

- 10 unidades (motores) con trayectorias completas run-to-failure
- 1 modo de fallo: degradación en eficiencia de la turbina de alta presión (HPT-E)
- Es el sub-dataset más simple y limpio del conjunto, por eso fue usado originalmente para diagnósticos basados en modelos físicos

Cada unidad (motor) tiene asignada una clase de vuelo fija. La clase 1 representa vuelos cortos (1–3 h) a baja altitud y velocidad; la clase 2 vuelos de duración media (3–5 h) a mayor altitud; la clase 3 los vuelos más largos (5–7 h) y de mayor altitud.

Debemos destacar que para el presente proyecto, nos enfocaremos en la clase de vuelos más largos (clase 3).

El archivo .h5 descargado de https://phm-datasets.s3.amazonaws.com/NASA/17.+Turbofan+Engine+Degradation+Simulation+Data+Set+2.zip contiene dos conjuntos de datos: el conjunto de desarrollo (train) y el conjunto de prueba (test), divididos según la siguiente estructura de bloques:

| Clave HDF5 | Descripción |
|---|---|
| `W_dev` / `W_test` | condiciones de vuelo |
| `X_s_dev` / `X_s_test` | Señales de sensores medidos |
| `Y_dev` / `Y_test` | Etiqueta RUL en ciclos (variable objetivo) |
| `A_dev` / `A_test` | Datos auxiliares (unidad, ciclo, clase de vuelo, estado de salud) |
| `W_var`, `X_s_var`, `A_var` | Nombres de las variables en cada bloque |

La descripción de las variables contenidas dentro de cada bloque es la siguiente:

Bloque `W` = condiciones operativas del vuelo en cada instante:

| # | Símbolo | Descripción | Unidades |
|---|---|---|---|
| 1 | `alt` | Altitud | ft |
| 2 | `Mach` | Número de Mach de vuelo | — |
| 3 | `TRA` | Ángulo del throttle-resolver | % |
| 4 | `T2` | Temperatura total en la entrada del ventilador | °R |

Bloque `X_s` = mediciones físicas de los sensores del motor:

| # | Símbolo | Descripción | Unidades |
|---|---|---|---|
| 1 | `Wf` | Flujo de combustible | pps |
| 2 | `Nf` | Velocidad física del ventilador | rpm |
| 3 | `Nc` | Velocidad física del núcleo | rpm |
| 4 | `T24` | Temperatura total salida LPC | °R |
| 5 | `T30` | Temperatura total salida HPC | °R |
| 6 | `T48` | Temperatura total salida HPT | °R |
| 7 | `T50` | Temperatura total salida LPT | °R |
| 8 | `P15` | Presión total en conducto de derivación (bypass) | psia |
| 9 | `P2` | Presión total entrada ventilador | psia |
| 10 | `P21` | Presión total salida ventilador | psia |
| 11 | `P24` | Presión total salida LPC | psia |
| 12 | `Ps30` | Presión estática salida HPC | psia |
| 13 | `P40` | Presión total salida cámara de combustión | psia |
| 14 | `P50` | Presión total salida LPT | psia |

Bloque `A` = datos auxiliares:

| # | Símbolo | Descripción |
|---|---|---|
| 1 | `unit` | Número de unidad (motor) |
| 2 | `cycle` | Número de ciclo de vuelo |
| 3 | `Fc` | Clase de vuelo (1, 2 o 3) |
| 4 | `hs` | Estado de salud del motor (health state) |


### Resultados:

#### Dataset y partición

El análisis se restringe a la clase de vuelo Fc=3, que en DS01 corresponde a 3 unidades (motores):

| Conjunto | Unidad(es) | Observaciones | Ciclos | RUL (ciclos) |
|---|---|---|---|---|
| Entrenamiento | 2 | 1.049.088 | 75 | 0 – 74 |
| Validación | 5 | 1.280.147 | 89 | 0 – 88 |
| Test | 10 | 1.190.670 | 82 | — |

El split sigue el esquema leave-one-unit-out: cada motor completo va íntegramente a un único conjunto, evitando cualquier fuga de información entre ciclos del mismo motor.

#### Feature Engineering

El proceso de feature engineering generó un total de **118 features** por observación, distribuidas en cuatro grupos:

| Grupo | Features |
|---|---|
| Raw sensores + condiciones de vuelo + posición | 20 |
| Rolling mean / std (ventanas 5 y 10 ciclos) | 56 |
| Slope inter-ciclo (ventana 10 ciclos) | 14 |
| Estadísticos intra-ciclo (max, std) | 28 |
| **TOTAL** | **118** |

#### Optimización de hiperparámetros

La búsqueda con Optuna (50 trials) redujo el RMSE de validación del baseline de **10.17** a **7.73** ciclos. Los hiperparámetros óptimos encontrados fueron:

| Hiperparámetro | Valor |
|---|---|
| `n_estimators` | 580 |
| `max_depth` | 4 |
| `learning_rate` | 0.1579 |
| `subsample` | 0.8345 |
| `colsample_bytree` | 0.9953 |
| `min_child_weight` | 18 |
| `reg_alpha` | 0.0078 |
| `reg_lambda` | 0.3191 |
| `gamma` | 0.7991 |

#### Métricas de evaluación

Las métricas se calculan sobre predicciones **agregadas por ciclo (mediana)**, siguiendo la arquitectura predecir-luego-agregar:

| Conjunto | RMSE | MAE | R² | Score NASA (S) |
|---|---|---|---|---|
| Entrenamiento | 0.003 | 0.002 | 1.0000 | 0.0 |
| Validación | 7.727 | 6.533 | 0.9095 | 65.7 |
| **Test** | **3.634** | **2.990** | **0.9764** | **22.5** |

El modelo explica el **97.6% de la varianza del RUL** en el conjunto de test, con un error típico de **3.6 ciclos** sobre un rango de RUL de 0 a 82 ciclos. El Score NASA de 22.5 en test indica una penalización asimétrica baja, con predominio de predicciones ligeramente adelantadas (conservadoras) frente al fallo.

#### Importancia de features

La feature más influyente es `cycle_norm` (posición normalizada del ciclo dentro de la vida del motor), que concentra el **96.3% del gain total**. Entre los sensores, los más informativos son las temperaturas de salida de la turbina de alta presión (`T48`) y del compresor de baja presión (`T24`), principalmente a través de sus estadísticos rolling inter-ciclo, lo que es coherente con el modo de fallo del dataset (degradación HPT-E).

#### Análisis de incertidumbre

La dispersión intra-ciclo de las predicciones individuales (`RUL_std`) permite construir un **mapa de alarma operativo** que clasifica cada ciclo en cuatro zonas según su RUL predicho e incertidumbre asociada, identificando los ciclos de alta urgencia (bajo RUL + alta incertidumbre) como candidatos prioritarios de intervención de mantenimiento.


### Discusión de resultados:

### Discusión de resultados:

#### Capacidad predictiva del modelo

Los resultados en el conjunto de test (RMSE=3.634, MAE=2.990, R²=0.9764) confirman que el modelo XGBoost es capaz de estimar el RUL con alta precisión sobre datos no vistos. Un error medio de aproximadamente 3 ciclos sobre un rango total de 82 ciclos representa un error relativo inferior al 4%, lo que es satisfactorio para un problema de regresión sobre señales de degradación real.

La diferencia entre los resultados de validación (RMSE=7.727, R²=0.9095) y test (RMSE=3.634, R²=0.9764) no indica un problema de generalización, sino que refleja la variabilidad natural entre unidades: cada motor tiene su propia trayectoria de degradación, y el modelo entrenado sobre la unidad 2 generaliza mejor a la unidad 10 (test) que a la unidad 5 (validación). Con solo 3 unidades disponibles en Fc=3, esta variabilidad inter-unidad es inevitable y esperada.

El sobreajuste en entrenamiento (RMSE=0.003, R²=1.0000) es consecuencia directa de entrenar sobre una única unidad: el modelo memoriza perfectamente la trayectoria de esa unidad. El criterio real de evaluación son los conjuntos de validación y test, donde el rendimiento es sólido.

#### Rol del feature engineering

La dominancia de `cycle_norm` (96.3% del gain) refleja que la posición normalizada del motor dentro de su vida útil es la señal más informativa para estimar el RUL, lo cual es coherente con la naturaleza del problema: a mayor avance en la vida del motor, menor RUL remanente por definición.

Entre los sensores físicos, `T48` (temperatura de salida de la turbina de alta presión) y `T24` (temperatura de salida del compresor de baja presión) son los más relevantes, siempre a través de sus estadísticos rolling inter-ciclo y no de sus valores instantáneos. Esto confirma que la **tendencia acumulada de degradación** entre vuelos es más informativa que el valor puntual de cada lectura, justificando el diseño del feature engineering adoptado.

#### Arquitectura predecir-luego-agregar

El hecho de que la mediana y la media por ciclo produzcan resultados prácticamente idénticos (RMSE=3.634 en ambos casos) indica que las predicciones individuales dentro de cada ciclo son consistentes y con poca dispersión. La arquitectura cumple su función: proporcionar una estimación robusta por ciclo a partir de miles de observaciones individuales, sin depender de una única predicción puntual.

#### Aplicabilidad al mantenimiento predictivo

Un error de ±3 ciclos en la estimación del RUL, sobre vuelos de clase 3 (5–7 horas), representa una ventana de predicción operativamente útil: permite planificar intervenciones de mantenimiento con suficiente antelación sin incurrir en paradas innecesarias prematuras. El mapa de alarma basado en `RUL_std` añade una capa adicional de información al identificar no solo cuándo intervenir, sino en qué ciclos la estimación es menos fiable, lo que es de valor directo para la toma de decisiones operativas.